#Chat Inspections

##Data Collection

In [1]:
!pip install textblob

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
  accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  roc_auc_score,
  classification_report
)

df = pd.read_parquet('https://huggingface.co/datasets/ddds-Capstone/Datasets/resolve/main/chat_metrics.parquet')


##TF-IDF
Term Frequency-Inverse Document Frequency

In [2]:
#Question TF-IDF
tfidf_question = TfidfVectorizer(max_features=500, stop_words='english')

question_tfidf = tfidf_question.fit_transform(df['question_en'].fillna(''))

question_features = tfidf_question.get_feature_names_out()

#Weight of each word across all questions
scores = np.asarray(question_tfidf.sum(axis=0)).flatten()

top_words = pd.DataFrame({'word': question_features, 'tfidf_score':scores}).sort_values('tfidf_score', ascending=False)

#Answer TF-IDF
tfidf_answer = TfidfVectorizer(max_features=500, stop_words='english')

answer_tfidf = tfidf_answer.fit_transform(df['answer_en'].fillna(''))

answer_features = tfidf_answer.get_feature_names_out()

#Weight of each word across all questions
scores = np.asarray(answer_tfidf.sum(axis=0)).flatten()

top_words = pd.DataFrame({'word': answer_features, 'tfidf_score':scores}).sort_values('tfidf_score', ascending=False)
top_words.head(20)

,word,tfidf_score
449,td,38.171470
264,information,27.945269
445,sunport,27.544662
71,albuquerque,24.184012
471,tsa,22.620104
58,abq,22.298030
453,terminal,21.776694
268,international,20.188348
256,https,19.798388
140,com,19.229386


##Sentiment Analysis

In [3]:
df['polarity'] = df['answer_en'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df['subjectivity'] = df['answer_en'].apply(lambda x: TextBlob(str(x)).sentiment.subjectivity)

df['polarity'] = df['question_en'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df['subjectivity'] = df['question_en'].apply(lambda x: TextBlob(str(x)).sentiment.subjectivity)

##Make a Copy

In [4]:
df_clean = df.copy()

##Data Cleaning

In [5]:
#Keep sessions with star_rating only
df_clean.dropna(subset=['star_rating'], inplace=True)

#Binary Satisfaction
df_clean['satisfaction'] = (df_clean['star_rating'] >= 4).astype(int)

df_clean['satisfaction'].value_counts(normalize=True)

#Set target
target = 'satisfaction'

##Processing

In [6]:
features = [

  #Time
  'day_sin',
  'day_cos',
  'hour_sin',
  'hour_cos',
  'month_sin',
  'month_cos',

  #Performance
  'processing_time_seconds',
  'total_tokens',
  'input_tokens',
  'output_tokens',
  'model_calls',
  'tool_calls_count',

  #Q&A
  'question_length',
  'answer_length',

  #Maps
  'has_geolocation',

  #Primary Category
  'primary_category_greetings',
  'primary_category_sunport_amenities',
  'primary_category_navigation',
  'primary_category_airline_logistics',
  'primary_category_general_info',

  #Selected Agent
  'selected_agentreporter',
  'selected_agentplanner',
  'selected_agentlocation',
  'selected_agentlocation_fallback_to_reporter',
  'selected_agentbroad_search_synthesis',
  'selected_agentbroad_search_passthrough',

  #Sentiment
  'polarity',
  'subjectivity'
]

X = df_clean[features]
y = df_clean['satisfaction']

###Train/test split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
  X,
  y,
  test_size=0.20,
  random_state=42,
  stratify=y #Maintains roughly same proportion
)

###Gaussian Naive Bayes

In [8]:
#Model
gnb = GaussianNB()

#Fit
gnb.fit(X_train, y_train)

#Predict
y_pred_gnb = gnb.predict(X_test)

#Probability
y_prob_gnb = gnb.predict_proba(X_test)[:,1]

#Evaluate
print('Accuracy:', accuracy_score(y_test, y_pred_gnb))
print('Precision:', precision_score(y_test, y_pred_gnb))
print('Recall:', recall_score(y_test, y_pred_gnb))
print('F1:', f1_score(y_test, y_pred_gnb))
print('ROC-AUC:', roc_auc_score(y_test, y_prob_gnb))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_gnb))

Accuracy: 0.5402298850574713
Precision: 0.6875
Recall: 0.2391304347826087
F1: 0.3548387096774194
ROC-AUC: 0.6826617179215271

Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.88      0.64        41
           1       0.69      0.24      0.35        46

    accuracy                           0.54        87
   macro avg       0.60      0.56      0.50        87
weighted avg       0.60      0.54      0.49        87

